# NB-D · InceptionTime Distilled — 154 Classes (GPU T4)

Knowledge Distillation: single InceptionTime student trained on soft labels from the 5-model ensemble.

- Loads pre-trained ensemble from `results/inc_154/` (read-only)
- Generates soft probability labels on training set
- Trains one student with KL-divergence loss + cosine LR decay (200 epoch cap)
- Target: Top-1 ≥ 0.356
- Saves dissertation model only if target met, never overwrites
- All outputs → `results/inc_154_distilled/`


In [1]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print('No GPU -- Runtime -> Change runtime type -> T4 GPU')
else:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.backend.clear_session()
with tf.device('/GPU:0'): _ = tf.constant([1.0])+tf.constant([2.0])
print('GPU ready')


GPU ready


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os, json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

PROJECT_DIR   = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2'
DATA_NPZ_DIR  = os.path.join(PROJECT_DIR, 'data', 'npz')
DATA_META_DIR = os.path.join(PROJECT_DIR, 'data', 'meta')
ENSEMBLE_DIR  = os.path.join(PROJECT_DIR, 'results', 'inc_154')
RESULTS_DIR   = os.path.join(PROJECT_DIR, 'results', 'inc_154_distilled')
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_NPZ = os.path.join(DATA_NPZ_DIR, 'ASL_154_train_cif.npz')
TEST_NPZ  = os.path.join(DATA_NPZ_DIR, 'ASL_154_test_cif.npz')
LE_PATH   = os.path.join(DATA_META_DIR, 'ASL_154_label_encoder.pkl')

SEEDS         = [42, 123, 456, 789, 1011]
N_ENSEMBLE    = len(SEEDS)
N_FILTERS     = 64
DEPTH         = 6
DROPOUT_BLOCK = 0.4
DROPOUT_HEAD  = 0.5
L2_REG        = 1e-3
BATCH_SIZE    = 64
EPOCHS        = 200
LR_RATE       = 5e-4
TEMPERATURE   = 3.0
STUDENT_SEED  = 42
TOP1_TARGET   = 0.356
MODEL_NAME    = 'InceptionTime 154-class Distilled'
MODEL_TAG     = 'inc_154_distilled'

print(f'Project   : {PROJECT_DIR}')
print(f'Ensemble  : {ENSEMBLE_DIR}')
print(f'Results   : {RESULTS_DIR}')


Project   : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2
Ensemble  : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/inc_154
Results   : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/inc_154_distilled


In [6]:
d_tr = np.load(TRAIN_NPZ, allow_pickle=True)
d_te = np.load(TEST_NPZ,  allow_pickle=True)
X_train = d_tr['X'].transpose(0,2,1).astype(np.float32)
X_test  = d_te['X'].transpose(0,2,1).astype(np.float32)
y_train = d_tr['y']
y_test  = d_te['y']
le      = joblib.load(LE_PATH)
N_CLASSES = len(le.classes_)
T, N_CH   = X_train.shape[1], X_train.shape[2]
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'T={T}  N_CH={N_CH}  Classes={N_CLASSES}')


Train: (926, 40, 126)  Test: (309, 40, 126)
T=40  N_CH=126  Classes=154


In [7]:
def inception_module(x, n_filters=64, kernel_sizes=(3,5,9),
                     bottleneck_size=32, dropout=0.4):
    inp = x
    if int(x.shape[-1]) > 1:
        x = tf.keras.layers.Conv1D(bottleneck_size,1,padding='same',use_bias=False)(x)
    branches=[tf.keras.layers.Conv1D(n_filters,ks,padding='same',use_bias=False)(x)
              for ks in kernel_sizes]
    mp=tf.keras.layers.MaxPool1D(3,strides=1,padding='same')(inp)
    mp=tf.keras.layers.Conv1D(n_filters,1,padding='same',use_bias=False)(mp)
    branches.append(mp)
    out=tf.keras.layers.Concatenate()(branches)
    out=tf.keras.layers.BatchNormalization()(out)
    out=tf.keras.layers.Activation('relu')(out)
    return tf.keras.layers.Dropout(dropout)(out)

def build_model(T,C,n_cls,seed,depth=6,n_filters=64):
    tf.keras.backend.clear_session(); tf.random.set_seed(seed)
    inputs=tf.keras.Input(shape=(T,C)); x=inputs; res=x
    for d in range(depth):
        x=inception_module(x,n_filters=n_filters,dropout=DROPOUT_BLOCK)
        if (d+1)%3==0:
            sc=tf.keras.layers.Conv1D(n_filters*4,1,padding='same',use_bias=False)(res)
            sc=tf.keras.layers.BatchNormalization()(sc)
            x=tf.keras.layers.Add()([x,sc])
            x=tf.keras.layers.Activation('relu')(x)
            res=x
    x=tf.keras.layers.GlobalAveragePooling1D()(x)
    x=tf.keras.layers.Dropout(DROPOUT_HEAD)(x)
    out=tf.keras.layers.Dense(n_cls,activation='softmax',
        kernel_regularizer=tf.keras.regularizers.l2(L2_REG))(x)
    return tf.keras.Model(inputs,out)

preview=build_model(T,N_CH,N_CLASSES,seed=STUDENT_SEED)
print(f'Params per model: {preview.count_params():,}')
del preview


Params per model: 489,434


In [8]:
print('Loading ensemble models (read-only)...')
ensemble_models=[]
for seed in SEEDS:
    mp=os.path.join(ENSEMBLE_DIR,f'best_model_seed{seed}.keras')
    if not os.path.exists(mp):
        raise FileNotFoundError(
            f'Ensemble model not found: {mp}\n'
            f'Run nbd_154_inceptiontime.ipynb first to train ensemble.')
    m=tf.keras.models.load_model(mp)
    ensemble_models.append(m)
    print(f'  loaded seed={seed}')
print(f'Ensemble ready ({len(ensemble_models)} models)')


Loading ensemble models (read-only)...
  loaded seed=42
  loaded seed=123
  loaded seed=456
  loaded seed=789
  loaded seed=1011
Ensemble ready (5 models)


In [9]:
SOFT_LABELS_PATH = os.path.join(RESULTS_DIR,'soft_labels_train.npy')
SOFT_TEST_PATH   = os.path.join(RESULTS_DIR,'soft_labels_test.npy')

def softmax_with_temperature(probs, T):
    '''Re-soften averaged ensemble probabilities using temperature T.'''
    log_p = np.log(np.clip(probs,1e-10,1.0)) / T
    log_p -= log_p.max(axis=-1,keepdims=True)
    e = np.exp(log_p)
    return (e / e.sum(axis=-1,keepdims=True)).astype(np.float32)

if os.path.exists(SOFT_LABELS_PATH):
    print('[SKIP] Loading cached soft labels...')
    soft_labels_train = np.load(SOFT_LABELS_PATH)
else:
    print(f'Generating soft labels on {len(X_train)} training samples (T={TEMPERATURE})...')
    all_preds=[]
    for i,m in enumerate(ensemble_models):
        p=m.predict(X_train,batch_size=64,verbose=0)
        all_preds.append(p)
        print(f'  member {i+1}/{N_ENSEMBLE} done')
    raw_avg=np.mean(all_preds,axis=0)
    soft_labels_train=softmax_with_temperature(raw_avg,TEMPERATURE)
    np.save(SOFT_LABELS_PATH,soft_labels_train)
    print(f'Saved -> {SOFT_LABELS_PATH}')

if not os.path.exists(SOFT_TEST_PATH):
    all_te=[m.predict(X_test,batch_size=64,verbose=0) for m in ensemble_models]
    np.save(SOFT_TEST_PATH, softmax_with_temperature(np.mean(all_te,axis=0),TEMPERATURE))
    print(f'Test soft labels saved -> {SOFT_TEST_PATH}')

print(f'soft_labels_train: {soft_labels_train.shape}  row-sum check: {soft_labels_train[0].sum():.4f}')


Generating soft labels on 926 training samples (T=3.0)...
  member 1/5 done
  member 2/5 done
  member 3/5 done
  member 4/5 done
  member 5/5 done
Saved -> /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/inc_154_distilled/soft_labels_train.npy
Test soft labels saved -> /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/inc_154_distilled/soft_labels_test.npy
soft_labels_train: (926, 154)  row-sum check: 1.0000


In [ ]:
STUDENT_PATH = os.path.join(RESULTS_DIR,'student_best.keras')
HISTORY_PATH = os.path.join(RESULTS_DIR,'student_history.json')

if os.path.exists(STUDENT_PATH):
    print(f'[SKIP] Loading existing student model from {STUDENT_PATH}')
    student=tf.keras.models.load_model(STUDENT_PATH)
else:
    student=build_model(T,N_CH,N_CLASSES,seed=STUDENT_SEED,depth=DEPTH,n_filters=N_FILTERS)
    steps_per_epoch=int(np.ceil(len(X_train)/BATCH_SIZE))
    lr_schedule=tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR_RATE,
        decay_steps=EPOCHS*steps_per_epoch,
        alpha=1e-6)
    student.compile(
        optimizer=tf.keras.optimizers.Adam(lr_schedule),
        loss=tf.keras.losses.KLDivergence(),
        metrics=['accuracy'])
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            STUDENT_PATH,save_best_only=True,monitor='accuracy',mode='max',verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='loss',patience=25,restore_best_weights=True,verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='loss',factor=0.5,patience=12,min_lr=1e-7,verbose=1),
    ]
    print(f'Training student: up to {EPOCHS} epochs, T={TEMPERATURE}, seed={STUDENT_SEED}')
    history=student.fit(
        X_train,soft_labels_train,
        batch_size=BATCH_SIZE,epochs=EPOCHS,
        callbacks=callbacks,verbose=1)
    best_acc=max(history.history['accuracy'])
    print(f'Best train acc: {best_acc:.4f}  Epochs: {len(history.history["accuracy"])}')
    with open(HISTORY_PATH,'w') as f:
        json.dump({k:[float(v) for v in vs] for k,vs in history.history.items()},f,indent=2)
    student=tf.keras.models.load_model(STUDENT_PATH)
    print('Student training complete.')


Training student: up to 200 epochs, T=3.0, seed=42
Epoch 1/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 810ms/step - accuracy: 0.0087 - loss: 0.9772
Epoch 1: accuracy improved from None to 0.00972, saving model to /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/inc_154_distilled/student_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/inc_154_distilled/student_best.keras
15/15 ━━━━━━━━━━━━━━━━━━━━ 34s 868ms/step - accuracy: 0.0097 - loss: 0.8451 - learning_rate: 4.9997e-04
Epoch 2/200
13/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0046 - loss: 0.6111
Epoch 2: accuracy did not improve from 0.00972
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0076 - loss: 0.5822 - learning_rate: 4.9988e-04
Epoch 3/200
13/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0139 - loss: 0.5022
Epoch 3: accuracy improved from 0.00972 to 0.01080, saving 

TypeError: This optimizer was created with a `LearningRateSchedule` object as its `learning_rate` constructor argument, hence its learning rate is not settable. If you need the learning rate to be settable, you should instantiate the optimizer with a float `learning_rate` argument.

In [ ]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

def full_eval(y_true, y_proba, le, model_name, model_tag, results_dir):
    os.makedirs(results_dir, exist_ok=True)
    N_CLS  = len(le.classes_)
    y_pred = np.argmax(y_proba, axis=1)
    n      = len(y_true)
    acc      = accuracy_score(y_true, y_pred)
    top5     = sum(y_true[k] in np.argsort(y_proba[k])[-5:] for k in range(n)) / n
    f1_mac   = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_wt    = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec_mac = precision_score(y_true, y_pred, average='macro',    zero_division=0)
    rec_mac  = recall_score(y_true, y_pred,    average='macro',    zero_division=0)
    metrics  = {'model':model_name,'top1_acc':float(acc),'top5_acc':float(top5),
                'f1_macro':float(f1_mac),'f1_weighted':float(f1_wt),
                'precision_macro':float(prec_mac),'recall_macro':float(rec_mac),
                'n_test':int(n),'n_classes':int(N_CLS)}
    with open(os.path.join(results_dir,f'{model_tag}_metrics.json'),'w') as f:
        json.dump(metrics,f,indent=2)
    report=classification_report(y_true,y_pred,
        target_names=[str(c) for c in le.classes_],output_dict=True,zero_division=0)
    with open(os.path.join(results_dir,f'{model_tag}_report.json'),'w') as f:
        json.dump(report,f,indent=2)
    # Confusion matrix
    cm=confusion_matrix(y_true,y_pred)
    n_show=min(50,cm.shape[0])
    fig,ax=plt.subplots(figsize=(14,12))
    im=ax.imshow(cm[:n_show,:n_show],cmap='Blues',aspect='auto')
    ax.set_title(f'{model_name}\nConfusion Matrix (top {n_show} classes)',fontweight='bold',fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ticks=np.arange(n_show); lbls=[str(le.classes_[i]) for i in range(n_show)]
    ax.set_xticks(ticks); ax.set_xticklabels(lbls,rotation=90,fontsize=6)
    ax.set_yticks(ticks); ax.set_yticklabels(lbls,fontsize=6)
    plt.colorbar(im,ax=ax,fraction=0.03); plt.tight_layout()
    plt.savefig(os.path.join(results_dir,f'{model_tag}_confusion_chart.png'),dpi=150)
    plt.show()
    pd.DataFrame(cm,index=[str(c) for c in le.classes_],
                 columns=[str(c) for c in le.classes_]).to_csv(
        os.path.join(results_dir,f'{model_tag}_confusion_table.csv'))
    # ROC
    y_bin=label_binarize(y_true,classes=np.arange(N_CLS))
    fpr_grid=np.linspace(0,1,300)
    tprs,aucs,cls_pos=[],[],[]
    for i in range(N_CLS):
        if y_bin[:,i].sum()==0: continue
        fpr,tpr,_=roc_curve(y_bin[:,i],y_proba[:,i])
        aucs.append(auc(fpr,tpr)); tprs.append(np.interp(fpr_grid,fpr,tpr)); cls_pos.append(i)
    mean_tpr=np.mean(tprs,axis=0); mean_tpr[0]=0.0; mean_tpr[-1]=1.0
    macro_auc=np.mean(aucs)
    sorted_a=sorted(zip(aucs,cls_pos),reverse=True)
    top5i=[x for _,x in sorted_a[:5]]; bot5i=[x for _,x in sorted_a[-5:]]
    fig,axes=plt.subplots(1,2,figsize=(16,6))
    ax=axes[0]
    ax.plot(fpr_grid,mean_tpr,color='navy',lw=2,label=f'Macro AUC={macro_auc:.3f}')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('FPR',fontsize=11); ax.set_ylabel('TPR',fontsize=11)
    ax.set_title(f'{model_name}\nMacro-Avg ROC',fontweight='bold')
    ax.legend(loc='lower right',fontsize=9); ax.grid(alpha=0.3)
    ax=axes[1]
    ct=['#1f77b4','#2ca02c','#d62728','#9467bd','#8c564b']
    cb=['#e377c2','#7f7f7f','#bcbd22','#17becf','#ff7f0e']
    for ci,col in zip(top5i,ct):
        fpr,tpr,_=roc_curve(y_bin[:,ci],y_proba[:,ci])
        ax.plot(fpr,tpr,color=col,lw=1.5,label=f'Top:{le.classes_[ci]} ({auc(fpr,tpr):.2f})')
    for ci,col in zip(bot5i,cb):
        fpr,tpr,_=roc_curve(y_bin[:,ci],y_proba[:,ci])
        ax.plot(fpr,tpr,color=col,lw=1.5,ls='--',label=f'Bot:{le.classes_[ci]} ({auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('FPR',fontsize=11); ax.set_ylabel('TPR',fontsize=11)
    ax.set_title('Best/Worst 5 Classes ROC',fontweight='bold')
    ax.legend(loc='lower right',fontsize=7); ax.grid(alpha=0.3)
    plt.suptitle(f'{model_name} | {N_CLS} classes | Macro AUC={macro_auc:.3f}',fontsize=11,y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir,f'{model_tag}_roc.png'),dpi=150,bbox_inches='tight')
    plt.show()
    metrics['macro_auc']=float(macro_auc)
    with open(os.path.join(results_dir,f'{model_tag}_metrics.json'),'w') as f:
        json.dump(metrics,f,indent=2)
    # Metrics bar
    fig,ax=plt.subplots(figsize=(10,5))
    mlbls=['Top-1\nAcc','Top-5\nAcc','Prec\n(mac)','Rec\n(mac)','F1\n(mac)','AUC\n(mac)']
    mvals=[acc,top5,prec_mac,rec_mac,f1_mac,macro_auc]
    cols=['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0','#00BCD4']
    bars=ax.bar(mlbls,mvals,color=cols,width=0.6)
    ax.set_ylim(0,min(1.0,max(mvals)*1.25+0.05))
    ax.axhline(TOP1_TARGET,color='red',ls='--',lw=1.5,label=f'Target Top-1 >= {TOP1_TARGET}')
    ax.set_title(f'{model_name} -- Test Metrics',fontweight='bold',fontsize=12)
    for bar,v in zip(bars,mvals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
    ax.legend(); ax.grid(axis='y',alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(results_dir,f'{model_tag}_metrics_bar.png'),dpi=150)
    plt.show()
    print(f'\n== {model_name} ==')
    print(f'  Top-1: {acc*100:.2f}%  Top-5: {top5*100:.2f}%')
    print(f'  Prec : {prec_mac*100:.2f}%  Rec: {rec_mac*100:.2f}%')
    print(f'  F1   : {f1_mac*100:.2f}%  AUC: {macro_auc:.4f}')
    return metrics

print('full_eval() defined')


In [ ]:
print('Predicting with student model...')
y_proba_student=student.predict(X_test,batch_size=64,verbose=0)
np.save(os.path.join(RESULTS_DIR,f'{MODEL_TAG}_proba_test.npy'),y_proba_student)
results=full_eval(y_test,y_proba_student,le,MODEL_NAME,MODEL_TAG,RESULTS_DIR)
print(f'All outputs saved -> {RESULTS_DIR}')


In [ ]:
TOP1=results['top1_acc']
DIS_MODEL_PATH=os.path.join(RESULTS_DIR,'dissertation_model_FINAL.keras')

print(f'\n{"="*55}')
print(f'  Top-1 Accuracy : {TOP1:.4f}')
print(f'  Target         : {TOP1_TARGET:.4f}')
print(f'{"="*55}')

if TOP1 >= TOP1_TARGET:
    if os.path.exists(DIS_MODEL_PATH):
        print(f'WARNING: {DIS_MODEL_PATH} already exists -- rename it first to save a new one.')
    else:
        student.save(DIS_MODEL_PATH)
        meta={'model_tag':MODEL_TAG,'top1_acc':float(TOP1),
              'top5_acc':float(results['top5_acc']),
              'f1_macro':float(results['f1_macro']),
              'macro_auc':float(results['macro_auc']),
              'n_classes':int(results['n_classes']),
              'temperature':TEMPERATURE,'student_seed':STUDENT_SEED,
              'epochs_config':EPOCHS,'batch_size':BATCH_SIZE,'lr_rate':LR_RATE,
              'architecture':f'InceptionTime DEPTH={DEPTH} N_FILTERS={N_FILTERS}',
              'distilled_from':f'{N_ENSEMBLE}-model ensemble seeds {SEEDS}'}
        with open(os.path.join(RESULTS_DIR,'dissertation_model_meta.json'),'w') as f:
            json.dump(meta,f,indent=2)
        print(f'TARGET MET -- Dissertation model saved:')
        print(f'  {DIS_MODEL_PATH}')
        print(f'  Meta -> dissertation_model_meta.json')
else:
    gap=TOP1_TARGET-TOP1
    print(f'Target NOT met (gap: {gap:.4f}). Tuning suggestions:')
    print('  1. Increase TEMPERATURE to 4.0 or 5.0 for softer labels')
    print('  2. Increase EPOCHS to 250')
    print('  3. Try STUDENT_SEED=123 or 456')
    print('  4. Lower DROPOUT_BLOCK to 0.3')
    print('  Student checkpoint kept at student_best.keras')


In [ ]:
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH) as f: hist=json.load(f)
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    axes[0].plot(hist['loss'],label='Train Loss')
    axes[0].set_title('Student -- KL Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist['accuracy'],label='Train Acc',color='#2196F3')
    axes[1].axhline(TOP1_TARGET,color='red',ls='--',lw=1.5,label=f'Target {TOP1_TARGET}')
    axes[1].set_title('Student -- Training Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.suptitle('Student Training Curves',fontweight='bold'); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR,f'{MODEL_TAG}_training_curves.png'),dpi=150)
    plt.show()
else:
    print('No history file (model loaded from cache). Skipping curves.')
